# 05 - Chignolin Acceleration CV Bottleneck

Train small shared CV bottlenecks to predict structural acceleration/dV. Then fit a linear readout from mean CV values on validation mutants only and evaluate it on test mutants.


In [ ]:
# Requires: pip install -e .

import itertools
import json
import random
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader, TensorDataset

from lss.models.cv_tm_predictor import BackboneModel, LinearBackboneModel
from lss.peptide import load_target_table
from lss.peptide_tm import (
    build_metric_dataset,
    build_train_val_test_split,
    extract_cv_sequences,
    metric_stats_from_train_mutants,
    prediction_stats,
    release_cuda_memory,
    run_sim_epoch,
)
from lss.plotting import PAPER_COLORS, apply_editorial_style, paper_caption, style_axes
from lss.utils import resolve_device

apply_editorial_style()




In [ ]:
# Config.

device = torch.device(resolve_device('cuda'))
seed = random.SystemRandom().randrange(1, 2**31)
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
print('device:', device)
print('seed:', seed)

backbone_type = 'linear'  # 'linear' or 'attention'
descriptor_token_dim = 32
target_metric_names = ['Tm', 'log_mfpt', 'log_mfpt_ratio', 'hlda_evalue']

history = 1
frames_per_traj = 50000
take_every_kth_frame = 10
train_mutant_count = 12
val_mutant_count = 12
cv_counts = (3, 4)
run_count = 5
hidden_size = 256
heads = 1
transformer_layers = 1
pre_pyramid_layers = 1
dropout = 0.001
batch_size = 64
weight_decay = 1e-5
sim_learning_rate = 1e-4
sim_epochs = 40
sim_early_stop_patience = 6
cls_weight = 0.2
time_lag_steps = 0

base_run_name = 'chignolin_05_accel_small_cv_mean_readout_multiseed'
ROOT = Path.cwd()
if not (ROOT / 'data').exists():
    ROOT = ROOT.parent
run_dir = ROOT / 'results' / base_run_name / 'run'
run_dir.mkdir(parents=True, exist_ok=True)

cfg = {
    'seed': seed,
    'history': history,
    'frames_per_traj': frames_per_traj,
    'take_every_kth_frame': take_every_kth_frame,
    'train_mutant_count': train_mutant_count,
    'val_mutant_count': val_mutant_count,
    'cv_counts': cv_counts,
    'run_count': run_count,
    'backbone_type': backbone_type,
    'hidden_size': hidden_size,
    'descriptor_token_dim': descriptor_token_dim,
    'linear_backbone_pooling': 'mean_std_max',
    'linear_backbone_descriptor_identity': True,
    'linear_backbone_token_mlp': 'linear_gelu_linear_gelu',
    'heads': heads,
    'transformer_layers': transformer_layers,
    'pre_pyramid_layers': pre_pyramid_layers,
    'dropout': dropout,
    'batch_size': batch_size,
    'weight_decay': weight_decay,
    'sim_learning_rate': sim_learning_rate,
    'sim_epochs': sim_epochs,
    'cls_weight': cls_weight,
    'time_lag_steps': time_lag_steps,
}
print(json.dumps({k: (list(v) if isinstance(v, tuple) else v) for k, v in cfg.items()}, indent=2))


In [ ]:
# Data.

packed = torch.load(ROOT / 'data' / 'peptide' / 'hlda_trajectories_compact.pt', map_location='cpu', weights_only=False)
x_all = packed.get('x_all', packed.get('x')).float()
time_all = packed.get('time_all', packed.get('time')).long()
offsets = packed.get('offsets', packed.get('traj_offsets')).long()
labels = packed['labels'].long()
mutants = packed['mutants']
feature_names = [str(name) for name in packed['feature_names']]
uniq_mutants_all = sorted(set(str(m) for m in mutants))
n_feat = int(x_all.shape[1])
distance_descriptor_indices = [i for i, name in enumerate(feature_names) if name.startswith('d') and name[1:].isdigit()]
distance_descriptor_names = [feature_names[i] for i in distance_descriptor_indices]
distance_descriptor_pairs = [(int(name[1]), int(name[2])) for name in distance_descriptor_names]
max_residue_index = max(max(pair) for pair in distance_descriptor_pairs)
distance_descriptor_metadata = torch.tensor(
    [
        [
            i / max_residue_index,
            j / max_residue_index,
            abs(j - i) / max_residue_index,
        ]
        for i, j in distance_descriptor_pairs
    ],
    dtype=torch.float32,
)

target_df = load_target_table(
    ROOT / 'data' / 'peptide' / 'Tm.csv',
    ROOT / 'data' / 'peptide' / 'mfpt_slice_thr0p34_tF0p25_tU0p57.csv',
    ROOT / 'data' / 'peptide' / 'hlda_evalues_thr0p34_tF0p25_tU0p57.csv',
)
target_df['log_mfpt'] = np.log(target_df['mfpt'].astype(float))
uniq_mutants = [m for m in uniq_mutants_all if m in set(target_df['mutant'].astype(str))]

print('features:', n_feat)
print('distance descriptors used as edge-like CV tokens:', len(distance_descriptor_indices), distance_descriptor_names)
print('edge-like metadata columns: residue_i_norm, residue_j_norm, sequence_separation_norm')
print('usable mutants:', len(uniq_mutants))
display(target_df.head())


In [ ]:
# Helpers.

def build_backbone(cv_count):
    token_sizes = (200, 80, 20, int(cv_count))
    if backbone_type == 'linear':
        return LinearBackboneModel(
            feat_dim=n_feat,
            history=history,
            hidden=hidden_size,
            token_sizes=token_sizes,
            dropout=dropout,
            descriptor_token_dim=descriptor_token_dim,
            descriptor_indices=distance_descriptor_indices,
            descriptor_metadata=distance_descriptor_metadata,
        ).to(device)
    if backbone_type == 'attention':
        return BackboneModel(
            feat_dim=n_feat,
            history=history,
            hidden=hidden_size,
            token_sizes=token_sizes,
            heads=heads,
            token_layers=transformer_layers,
            dropout=dropout,
            pre_pyramid_layers=pre_pyramid_layers,
            linear_cv_decoder=True,
        ).to(device)
    raise ValueError(f'Unknown backbone_type={backbone_type!r}')


def descriptor_residue_indices(feature_name):
    name = str(feature_name)
    if name.startswith('d') and name[1:].isdigit():
        return [int(ch) for ch in name[1:]]
    return []


def aggregate_residue_weights(feature_names, weights):
    rows = []
    for feature_name, weight in zip(feature_names, np.asarray(weights, dtype=float)):
        for residue_index in descriptor_residue_indices(feature_name):
            rows.append({'residue_index': residue_index, 'abs_weight': abs(float(weight))})
    return pd.DataFrame(rows).groupby('residue_index', as_index=False)['abs_weight'].sum().sort_values('abs_weight', ascending=False)


def _corr(x, y, method='pearson'):
    clean = pd.DataFrame({'x': x, 'y': y}).dropna()
    if len(clean) < 3 or clean['x'].std() < 1e-12 or clean['y'].std() < 1e-12:
        return np.nan
    return float(clean['x'].corr(clean['y'], method=method))


def _r2(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    denom = float(np.sum((y_true - y_true.mean()) ** 2))
    return float(1.0 - np.sum((y_true - y_pred) ** 2) / denom)


def mean_cv_cols(cv_count):
    return [f'cv{j}_mean' for j in range(int(cv_count))]


def summarize_cv_by_mutant(backbone, x_tensor, samples_df, split_name, cv_count):
    cv_sequences = extract_cv_sequences(backbone, x_tensor, samples_df, batch_size=batch_size, device=device)
    rows = []
    frame_rows = []
    sample_state = samples_df.reset_index(drop=True).copy()
    sample_state['state_label'] = sample_state['traj_id'].map({int(i): float(labels[i]) for i in range(len(labels))})

    for mutant, seq in cv_sequences.items():
        sub = sample_state[sample_state['mutant'].astype(str).eq(str(mutant))].copy().reset_index(drop=True)
        cv = seq.detach().cpu().numpy().astype(float)[:len(sub)]
        row = {'split': split_name, 'mutant': str(mutant)}
        frame_cols = ['mutant', 'traj_id', 't', 'state_label']
        for j in range(int(cv_count)):
            col = f'cv{j}'
            sub[col] = cv[:, j]
            row[f'{col}_mean'] = float(sub[col].mean())
            frame_cols.append(col)
        sub['cv'] = sub['cv0']
        frame_rows.append(sub[frame_cols + ['cv']].assign(split=split_name))
        rows.append(row)

    return pd.DataFrame(rows).merge(target_df, on='mutant', how='left'), pd.concat(frame_rows, ignore_index=True)


def fit_mean_cv_readout(val_df, eval_df, metric, cv_count):
    cols = mean_cv_cols(cv_count)
    x_val = val_df[cols].to_numpy(float)
    y_val = val_df[metric].to_numpy(float)
    x_eval = eval_df[cols].to_numpy(float)
    y_eval = eval_df[metric].to_numpy(float)
    x_mean = x_val.mean(axis=0)
    x_std = x_val.std(axis=0) + 1e-12
    x_val = (x_val - x_mean) / x_std
    x_eval = (x_eval - x_mean) / x_std
    coef, *_ = np.linalg.lstsq(np.column_stack([np.ones(len(x_val)), x_val]), y_val, rcond=None)
    pred = np.column_stack([np.ones(len(x_eval)), x_eval]) @ coef
    formula = '; '.join([f'intercept={coef[0]:.4g}'] + [f'{c}={w:.4g}' for c, w in zip(cols, coef[1:])])
    return y_eval, pred, formula


def mean_cv_readout_rows(val_df, test_df, epoch, cv_count, run_idx, run_seed):
    rows = []
    for metric in target_metric_names:
        for split_name, eval_df in [('val', val_df), ('test', test_df)]:
            y, pred, formula = fit_mean_cv_readout(val_df, eval_df, metric, cv_count)
            rows.append({
                'run_idx': int(run_idx),
                'run_seed': int(run_seed),
                'cv_count': int(cv_count),
                'epoch': int(epoch),
                'metric': metric,
                'split': split_name,
                'readout': 'mean_cv_linear_combo',
                'features': ','.join(mean_cv_cols(cv_count)),
                'formula': formula,
                'r2': _r2(y, pred),
                'pearson': _corr(pd.Series(pred), pd.Series(y), method='pearson'),
                'spearman': _corr(pd.Series(pred), pd.Series(y), method='spearman'),
                'mse': float(np.mean((y - pred) ** 2)),
                'n': int(len(y)),
            })
    return pd.DataFrame(rows)


def validation_selected_epoch_rows(readout_df):
    rows = []
    for (run_idx, cv_count, metric), group in readout_df.groupby(['run_idx', 'cv_count', 'metric'], sort=False):
        best_val = group[group['split'].eq('val')].sort_values('r2', ascending=False).iloc[0]
        test_row = group[group['split'].eq('test') & group['epoch'].eq(best_val['epoch'])].iloc[0].copy()
        test_row['selected_val_r2'] = best_val['r2']
        rows.append(test_row)
    return pd.DataFrame(rows)


In [ ]:
# Multi-seed small-CV acceleration bottleneck experiment.

config_json = run_dir / 'config.json'
config_json.write_text(json.dumps({k: (list(v) if isinstance(v, tuple) else v) for k, v in cfg.items()}, indent=2))

all_history = []
all_readouts = []
final_summary_parts = []
final_frame_parts = []
best_overall = {'tm_test_r2': -np.inf}

for cv_count in cv_counts:
    for run_idx in range(1, run_count + 1):
        run_seed = int(seed + 1000 * cv_count + run_idx)
        random.seed(run_seed)
        np.random.seed(run_seed)
        torch.manual_seed(run_seed)

        split_payload = build_train_val_test_split(
            run_seed=run_seed,
            uniq_mutants=uniq_mutants,
            train_mutant_count=train_mutant_count,
            val_mutant_count=val_mutant_count,
            mutants=mutants,
            history=history,
            time_lag_steps=time_lag_steps,
            frames_per_traj=frames_per_traj,
            take_every_kth_frame=take_every_kth_frame,
            x_all=x_all,
            time_all=time_all,
            offsets=offsets,
            labels=labels,
            n_feat=n_feat,
        )
        print(f'\nCV{cv_count} run {run_idx}/{run_count} seed={run_seed}')
        print('train/val/test mutants:', len(split_payload['train_mutants']), len(split_payload['val_mutants']), len(split_payload['test_mutants']))

        release_cuda_memory()
        backbone = build_backbone(cv_count)
        opt = torch.optim.AdamW(backbone.parameters(), lr=sim_learning_rate, weight_decay=weight_decay)
        train_loader = DataLoader(TensorDataset(split_payload['train_x'], split_payload['train_dv'], split_payload['train_cls']), batch_size=batch_size, shuffle=True, drop_last=False)
        val_loader = DataLoader(TensorDataset(split_payload['val_x'], split_payload['val_dv'], split_payload['val_cls']), batch_size=batch_size, shuffle=False, drop_last=False)
        test_loader = DataLoader(TensorDataset(split_payload['test_x'], split_payload['test_dv'], split_payload['test_cls']), batch_size=batch_size, shuffle=False, drop_last=False)

        best_val_loss = float('inf')
        best_state = None
        best_epoch = 0
        bad_epochs = 0

        for epoch in range(1, sim_epochs + 1):
            train_info = run_sim_epoch(backbone, train_loader, device=device, cls_weight=cls_weight, opt=opt)
            val_info = run_sim_epoch(backbone, val_loader, device=device, cls_weight=cls_weight, opt=None)
            test_info = run_sim_epoch(backbone, test_loader, device=device, cls_weight=cls_weight, opt=None)
            val_summary, _ = summarize_cv_by_mutant(backbone, split_payload['val_x'], split_payload['val_samples_df'], 'val', cv_count)
            test_summary, _ = summarize_cv_by_mutant(backbone, split_payload['test_x'], split_payload['test_samples_df'], 'test', cv_count)
            readout_rows = mean_cv_readout_rows(val_summary, test_summary, epoch, cv_count, run_idx, run_seed)
            all_readouts.append(readout_rows)

            tm_test = readout_rows[(readout_rows['metric'].eq('Tm')) & (readout_rows['split'].eq('test'))].iloc[0]
            log_test = readout_rows[(readout_rows['metric'].eq('log_mfpt')) & (readout_rows['split'].eq('test'))].iloc[0]
            all_history.append({
                'run_idx': run_idx,
                'run_seed': run_seed,
                'cv_count': cv_count,
                'epoch': epoch,
                **{f'train_{k}': v for k, v in train_info.items()},
                **{f'val_{k}': v for k, v in val_info.items()},
                **{f'test_{k}': v for k, v in test_info.items()},
                'tm_test_r2': tm_test['r2'],
                'tm_test_pearson': tm_test['pearson'],
                'log_mfpt_test_r2': log_test['r2'],
                'log_mfpt_test_pearson': log_test['pearson'],
            })

            print(
                f"CV{cv_count} run {run_idx:02d} epoch {epoch:03d} "
                f"val_loss={val_info['sim_loss']:.4g} test_loss={test_info['sim_loss']:.4g} "
                f"Tm test R2={tm_test['r2']:.3f} r={tm_test['pearson']:.3f} "
                f"logMFPT test R2={log_test['r2']:.3f} r={log_test['pearson']:.3f}"
            )

            if val_info['sim_loss'] < best_val_loss - 1e-5:
                best_val_loss = val_info['sim_loss']
                best_state = {k: v.detach().cpu().clone() for k, v in backbone.state_dict().items()}
                best_epoch = epoch
                bad_epochs = 0
            else:
                bad_epochs += 1
                if bad_epochs >= sim_early_stop_patience:
                    break

        backbone.load_state_dict(best_state)
        backbone.eval()
        final_val_summary, _ = summarize_cv_by_mutant(backbone, split_payload['val_x'], split_payload['val_samples_df'], 'val', cv_count)
        final_test_summary, final_test_frames = summarize_cv_by_mutant(backbone, split_payload['test_x'], split_payload['test_samples_df'], 'test', cv_count)
        final_readouts = mean_cv_readout_rows(final_val_summary, final_test_summary, best_epoch, cv_count, run_idx, run_seed)
        final_readouts['selection'] = 'best_structural_val_loss'
        final_summary = pd.concat([final_val_summary, final_test_summary], ignore_index=True)
        final_summary['run_idx'] = run_idx
        final_summary['run_seed'] = run_seed
        final_summary['cv_count'] = cv_count
        final_frame = final_test_frames.copy()
        final_frame['run_idx'] = run_idx
        final_frame['run_seed'] = run_seed
        final_frame['cv_count'] = cv_count
        final_summary_parts.append(final_summary)
        final_frame_parts.append(final_frame)
        all_readouts.append(final_readouts)

        tm_final = final_readouts[(final_readouts['metric'].eq('Tm')) & (final_readouts['split'].eq('test'))].iloc[0]
        if float(tm_final['r2']) > best_overall['tm_test_r2']:
            best_overall = {
                'tm_test_r2': float(tm_final['r2']),
                'cv_count': cv_count,
                'run_idx': run_idx,
                'run_seed': run_seed,
                'best_epoch': best_epoch,
                'backbone': backbone,
                'split_payload': split_payload,
                'summary': final_summary,
                'frames': final_frame,
                'readouts': final_readouts,
            }

sim_history_df = pd.DataFrame(all_history)
cv_readout_df = pd.concat(all_readouts, ignore_index=True)
final_cv_summary_df = pd.concat(final_summary_parts, ignore_index=True)
final_cv_frames_df = pd.concat(final_frame_parts, ignore_index=True)
best_backbone = best_overall['backbone']
best_cv_count = best_overall['cv_count']
best_run_idx = best_overall['run_idx']

sim_history_df.to_csv(run_dir / 'small_cv_sim_history.csv', index=False)
cv_readout_df.to_csv(run_dir / 'small_cv_mean_readout_by_epoch.csv', index=False)
final_cv_summary_df.to_csv(run_dir / 'small_cv_final_mutant_summaries.csv', index=False)
final_cv_frames_df.to_csv(run_dir / 'small_cv_final_test_frame_values.csv', index=False)
print('saved results to:', run_dir)
print('best structural-checkpoint Tm test R2:', best_overall['tm_test_r2'], 'CV', best_cv_count, 'run', best_run_idx)


In [ ]:
# Aggregate validation-selected test performance.

epoch_readout_df = cv_readout_df[cv_readout_df['selection'].isna()].copy()
selected_epoch_df = validation_selected_epoch_rows(epoch_readout_df)
selected_epoch_df.to_csv(run_dir / 'small_cv_validation_selected_test_readouts.csv', index=False)

plot_df = selected_epoch_df[selected_epoch_df['metric'].isin(['Tm', 'log_mfpt', 'hlda_evalue'])].copy()
fig, axes = plt.subplots(1, 3, figsize=(13.5, 3.8), sharey=False, constrained_layout=True)
for ax, metric in zip(axes, ['Tm', 'log_mfpt', 'hlda_evalue']):
    sub = plot_df[plot_df['metric'].eq(metric)]
    for cv_count, group in sub.groupby('cv_count', sort=True):
        ax.scatter(group['selected_val_r2'], group['r2'], s=42, alpha=0.85, label=f'CV{cv_count}')
    ax.axhline(0, color=PAPER_COLORS['ink'], lw=0.8, alpha=0.5)
    ax.set_title(f'{metric}: validation-selected epoch')
    ax.set_xlabel('validation R2')
    ax.set_ylabel('test R2')
    ax.grid(alpha=0.25)
axes[-1].legend(frameon=False)
fig.savefig(run_dir / 'small_cv_validation_selected_test_r2.png', dpi=200, bbox_inches='tight')
plt.show()

summary = (
    selected_epoch_df.groupby(['cv_count', 'metric'], as_index=False)
    .agg(test_r2_mean=('r2', 'mean'), test_r2_std=('r2', 'std'), test_pearson_mean=('pearson', 'mean'), n_runs=('run_idx', 'nunique'))
    .sort_values(['metric', 'test_r2_mean'], ascending=[True, False])
)
display(summary.round(4))

tm_best = selected_epoch_df[selected_epoch_df['metric'].eq('Tm')].sort_values('r2', ascending=False)
display(tm_best[['cv_count', 'run_idx', 'epoch', 'selected_val_r2', 'r2', 'pearson', 'spearman', 'mse', 'formula']].head(10).round(4))


In [ ]:
# Best-run test scatter for the mean-CV linear readout.

best_val_summary = best_overall['summary'][best_overall['summary']['split'].eq('val')].copy()
best_test_summary = best_overall['summary'][best_overall['summary']['split'].eq('test')].copy()
metrics_to_plot = ['Tm', 'log_mfpt', 'hlda_evalue']
fig, axes = plt.subplots(1, len(metrics_to_plot), figsize=(4.5 * len(metrics_to_plot), 4.0), constrained_layout=True)
for ax, metric in zip(axes, metrics_to_plot):
    y_test, pred_test, formula = fit_mean_cv_readout(best_val_summary, best_test_summary, metric, best_cv_count)
    r2 = _r2(y_test, pred_test)
    ax.scatter(y_test, pred_test, s=38, alpha=0.85, color=PAPER_COLORS['blue'])
    for i, (_, row) in enumerate(best_test_summary.iterrows()):
        ax.text(y_test[i], pred_test[i], row['mutant'], fontsize=6, alpha=0.65)
    lo = float(np.nanmin([np.min(y_test), np.min(pred_test)]))
    hi = float(np.nanmax([np.max(y_test), np.max(pred_test)]))
    ax.plot([lo, hi], [lo, hi], color=PAPER_COLORS['ink'], lw=1, ls='--')
    ax.set_title(f'{metric}: CV{best_cv_count} run {best_run_idx}, test R2={r2:.3f}')
    ax.set_xlabel(f'true {metric}')
    ax.set_ylabel(f'val-fit predicted {metric}')
    ax.grid(alpha=0.25)
fig.savefig(run_dir / 'small_cv_best_run_test_scatter.png', dpi=200, bbox_inches='tight')
plt.show()

paper_caption('Best-run validation-fitted formulas')
display(best_overall['readouts'][['metric', 'split', 'r2', 'pearson', 'spearman', 'mse', 'formula']].round(4))


In [ ]:
# CV0 traces for the best run, colored by folded/unfolded basin label.

plot_split = 'test'
trace_df = best_overall['frames'].copy()
plot_mutants = list(trace_df['mutant'].drop_duplicates()[:6])
state_colors = {0.0: PAPER_COLORS['blue'], 1.0: PAPER_COLORS['red']}
fig, axes = plt.subplots(len(plot_mutants), 1, figsize=(8.8, 1.65 * len(plot_mutants)), sharex=False, constrained_layout=True)
if len(plot_mutants) == 1:
    axes = [axes]
for ax, mutant in zip(axes, plot_mutants):
    sub = trace_df[trace_df['mutant'].eq(mutant)].sort_values(['traj_id', 't']).copy()
    for state_label, state_sub in sub.groupby('state_label', sort=False):
        ax.scatter(
            state_sub['t'],
            state_sub['cv0'],
            s=9,
            alpha=0.55,
            color=state_colors.get(float(state_label), '#777777'),
            label='folded' if state_label < 0.5 else 'unfolded',
        )
    ax.set_ylabel('CV0')
    ax.set_title(f'CV{best_cv_count} run {best_run_idx}: mutant {mutant}')
    ax.grid(alpha=0.22)
    ax.legend(frameon=False, loc='best')
axes[-1].set_xlabel('frame')
fig.suptitle('Best-run CV0 over frames, colored by basin label', y=1.01)
fig.savefig(run_dir / 'small_cv_best_run_cv0_trace_by_basin.png', dpi=200, bbox_inches='tight')
plt.show()


In [ ]:
# Decoder weights for the best run: dV contribution from each CV coordinate, no bias.

W = best_backbone.dv_head.weight.detach().cpu().numpy()
for cv_idx in range(best_cv_count):
    residue_weight_df = aggregate_residue_weights(feature_names, W[:, cv_idx])
    paper_caption(f'Best-run decoder weights: dV contribution from CV{cv_idx}')
    display(residue_weight_df.round(5).head(10))

    fig, ax = plt.subplots(1, 1, figsize=(5.8, 3.2))
    residue_plot = residue_weight_df.sort_values('residue_index')
    ax.bar(residue_plot['residue_index'].astype(str), residue_plot['abs_weight'], color=PAPER_COLORS['blue'])
    style_axes(ax, xlabel='residue index', ylabel=f'summed |CV{cv_idx} decoder weight|')
    plt.tight_layout()
    plt.show()


## Notes

- Training uses only structural acceleration/dV prediction plus the folded/unfolded auxiliary classifier from 02.
- `Tm`, `log_mfpt`, `log_mfpt_ratio`, and `hlda_evalue` are not used in training.
- The external readout is deliberately simple: fit a linear combination of mean CV values on validation mutants, then evaluate on test mutants.
- The notebook repeats CV3 and CV4 across several random train/val/test splits and reports mean/stdev test R2.
